# Data Cleaning - Club Piscine MMM

This notebook extracts and cleans data from the client's Excel files for use in the Marketing Mix Model.

**Files processed:**
1. Rapport de soumissions 2024 & 2025 - Quote requests by product type
2. Budget 2024 & 2025 - Media spend by channel
3. Recap Tableau Medias 2025 - Campaign performance metrics
4. Calendrier Fiscal - Fiscal calendar reference

**Note:** Blank cells represent missing data (client confirmed) - we preserve these as NaN without imputation.

---

## Media Channel Grouping Strategy (v2)

To optimize for limited historical data (~24 monthly observations), we consolidate media channels into **7 strategic groups**:

| Group | Budget Source Channels | Rationale |
|-------|------------------------|----------|
| **Television** | TELEVISION | Major investment, long carryover effect |
| **Audio** | RADIO + RADIO NUMÉRIQUE | Similar consumption patterns |
| **Affichage** | PANNEAUX ET AFFICHAGES NUMÉRIQUES | Outdoor exposure |
| **Search** | GOOGLE ADS + GOOGLE SHOPPING | Immediate intent, short memory |
| **Digital_Branding** | BANNIÈRES WEB PREMIUM + PREROLL PREMIUM + LAPRESSE + CONTENU DE MARQUE | Premium digital brand building |
| **Circulaire_Digital** | CIRCULAIRE DIGITALE | Promotional flyers |
| **Social_Media** | FACEBOOK (PROMO+PRODUIT) + INSTAGRAM (PROMO+PRODUIT) + PINTEREST + TIKTOK | Social platforms |

**Excluded channels:** PROGRAMMATIQUE, AUDIO ET PODCAST, ENVOIS POSTAUX

In [ ]:
# Cell 1: Imports and Setup
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

# Paths - adjust based on where this notebook is run
project_root = Path().cwd().parent if Path().cwd().name == 'notebooks' else Path().cwd()
raw_path = project_root / 'data' / 'raw'
processed_path = project_root / 'data' / 'processed'

# Create processed directory if it doesn't exist
processed_path.mkdir(parents=True, exist_ok=True)

print(f"Project root: {project_root}")
print(f"Raw data path: {raw_path}")
print(f"Processed data path: {processed_path}")

# List raw files
print("\nRaw files found:")
for f in raw_path.glob('*'):
    print(f"  - {f.name}")

---
## 1. Rapport de Soumissions (Quote Requests) - 2022 to 2025

Extracts monthly quote requests by product type from the RECAP sheet.

**IMPORTANT:** Each file contains multiple years of data side by side!

**2024 file structure:**
- Row 16: Year headers (2024, 2023, 2022 for each category)
- Rows 17-28: Monthly data
- Contains data for: 2024, 2023, 2022

**2025 file structure:**
- Row 19: Year headers (2025, 2024, 2023 for each category)
- Rows 20-31: Monthly data
- Contains data for: 2025, 2024, 2023 (and some 2022)

**Categories extracted:**
- Piscines Hors Terre (above-ground pools)
- Piscines Creusées (in-ground pools)
- Spas
- Autres Produits (other products)
- Services
- Autre (other)

**Data combination strategy:**
- 2025: from 2025 file
- 2024: from 2025 file (more recent)
- 2023: from 2025 file (more recent)
- 2022: from 2024 file

In [ ]:
# Cell 2: Clean Rapport de Soumissions (Quotes) - Multi-year extraction

def extract_soumissions_data(raw_path):
    """
    Extract quote data from both 2024 and 2025 Rapport de Soumissions files.
    
    Uses intelligent merging:
    - 2025 file is PRIMARY (more recent data)
    - Falls back to 2024 file for gaps
    - Preserves NaN for genuinely missing data
    """
    # Load both files
    df_2024_file = pd.read_excel(raw_path / 'Rapport de soumissions 2024.xlsx', 
                                  sheet_name='RECAP', header=None)
    df_2025_file = pd.read_excel(raw_path / 'Rapport de soumissions 2025.xlsx', 
                                  sheet_name='RECAP', header=None)
    
    months = ['Janvier', 'Février', 'Mars', 'Avril', 'Mai', 'Juin', 
              'Juillet', 'Août', 'Septembre', 'Octobre', 'Novembre', 'Décembre']
    
    # Column mappings for 2024 file (data starts at row 17)
    # Format: category: {year: column_index}
    cols_2024_file = {
        'piscines_hors_terre': {2024: 2, 2023: 3, 2022: 4},
        'piscines_creusees': {2024: 6, 2023: 7, 2022: 8},
        'spas': {2024: 10, 2023: 11, 2022: 12},
        'autres_produits': {2024: 14, 2023: 15, 2022: 16},
        'services': {2024: 18, 2023: 19, 2022: 20},
        'autre': {2024: 22, 2023: 23, 2022: 24},
    }
    
    # Column mappings for 2025 file (data starts at row 20)
    cols_2025_file = {
        'piscines_hors_terre': {2025: 2, 2024: 3, 2023: 4},  # Note: 2024 col (3) is empty!
        'piscines_creusees': {2024: 6, 2023: 7, 2022: 8},    # No 2025 column
        'spas': {2025: 10, 2024: 11, 2023: 12},
        'autres_produits': {2024: 18, 2023: 19, 2022: 20},   # No 2025 column
        'services': {2024: 22, 2023: 23, 2022: 24},          # No 2025 column
        'autre': {2024: 26, 2023: 27, 2022: 28},             # No 2025 column
    }
    
    all_data = []
    
    for year in [2022, 2023, 2024, 2025]:
        for i, month in enumerate(months):
            row_2024_file = 17 + i  # Data rows in 2024 file
            row_2025_file = 20 + i  # Data rows in 2025 file
            
            row_data = {'year': year, 'month': month, 'month_num': i + 1}
            
            for category in ['piscines_hors_terre', 'piscines_creusees', 'spas', 
                            'autres_produits', 'services', 'autre']:
                
                value = np.nan
                
                # Try 2025 file first (more recent)
                if category in cols_2025_file and year in cols_2025_file[category]:
                    col_idx = cols_2025_file[category][year]
                    if col_idx < df_2025_file.shape[1]:
                        val = df_2025_file.iloc[row_2025_file, col_idx]
                        if pd.notna(val):
                            value = val
                
                # If still NaN, try 2024 file
                if pd.isna(value):
                    if category in cols_2024_file and year in cols_2024_file[category]:
                        col_idx = cols_2024_file[category][year]
                        if col_idx < df_2024_file.shape[1]:
                            val = df_2024_file.iloc[row_2024_file, col_idx]
                            if pd.notna(val):
                                value = val
                
                row_data[category] = value
            
            all_data.append(row_data)
    
    df = pd.DataFrame(all_data)
    
    # Convert to numeric
    numeric_cols = ['piscines_hors_terre', 'piscines_creusees', 'spas', 
                    'autres_produits', 'services', 'autre']
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    return df

# Extract data
soumissions_combined = extract_soumissions_data(raw_path)

# Add totals
soumissions_combined['total_main_quotes'] = (
    soumissions_combined['piscines_hors_terre'].fillna(0) + 
    soumissions_combined['piscines_creusees'].fillna(0) + 
    soumissions_combined['spas'].fillna(0)
)
soumissions_combined['total_all_quotes'] = (
    soumissions_combined['total_main_quotes'] +
    soumissions_combined['autres_produits'].fillna(0) +
    soumissions_combined['services'].fillna(0) +
    soumissions_combined['autre'].fillna(0)
)

print("=" * 60)
print("SOUMISSIONS (QUOTES) - EXTRACTED")
print("=" * 60)
print(f"Shape: {soumissions_combined.shape}")
print(f"\nYears available: {sorted(soumissions_combined['year'].unique())}")

print("\nData availability by year:")
for year in [2022, 2023, 2024, 2025]:
    year_data = soumissions_combined[soumissions_combined['year'] == year]
    non_null = year_data[['piscines_hors_terre', 'piscines_creusees', 'spas']].notna().sum().sum()
    total = year_data[['piscines_hors_terre', 'piscines_creusees', 'spas']].size
    print(f"  {year}: {non_null}/{total} main product data points")

---
## 2. Budget Media Spend - With 7-Channel Grouping

Extracts monthly media spend and groups into 7 strategic channels for MMM:

1. **Television** - TV spend
2. **Audio** - Radio + Radio Numérique
3. **Affichage** - Panneaux et Affichages Numériques
4. **Search** - Google Ads + Google Shopping
5. **Digital_Branding** - Bannières Web Premium, Preroll Premium, La Presse, Contenu de Marque
6. **Circulaire_Digital** - Digital flyers
7. **Social_Media** - Facebook (Promo+Produit) + Instagram (Promo+Produit) + Pinterest + TikTok

**Excluded:** PROGRAMMATIQUE, AUDIO ET PODCAST, ENVOIS POSTAUX

In [ ]:
# Cell 3: Clean Budget Media Spend with 7-Channel Grouping

def clean_budget_grouped(file_path, year):
    """
    Clean the budget file and group channels into 7 strategic categories.
    
    Channel grouping:
    - Television: TELEVISION
    - Audio: RADIO + RADIO NUMÉRIQUE
    - Affichage: PANNEAUX ET AFFICHAGES NUMÉRIQUES
    - Search: GOOGLE ADS + GOOGLE SHOPPING
    - Digital_Branding: BANNIÈRES WEB PREMIUM + PREROLL PREMIUM + LAPRESSE + CONTENU DE MARQUE
    - Circulaire_Digital: CIRCULAIRE DIGITALE
    - Social_Media: FACEBOOK + INSTAGRAM + PINTEREST + TIKTOK
    
    Excluded: PROGRAMMATIQUE, AUDIO ET PODCAST, ENVOIS POSTAUX
    """
    df_raw = pd.read_excel(file_path, sheet_name=0, header=None)
    
    # Define channel grouping mapping
    CHANNEL_GROUPS = {
        # Television
        'TELEVISION': 'Television',
        
        # Audio
        'RADIO': 'Audio',
        'RADIO NUMÉRIQUE': 'Audio',
        
        # Affichage
        'PANNEAUX': 'Affichage',
        'PANNEAUX ET AFFICHAGES NUMÉRIQUES': 'Affichage',
        
        # Search (Google)
        'GOOGLE ADS': 'Search',
        'GOOGLE DISPLAY + PREROLL': 'Search',
        'GOOGLE SHOPPING': 'Search',
        
        # Digital Branding
        'BANNIÈRES WEB - PREMIUM': 'Digital_Branding',
        'PREROLL - PREMIUM': 'Digital_Branding',
        'LAPRESSE (LP+, PREROLL, DISPLAY)': 'Digital_Branding',
        'CONTENU DE MARQUE': 'Digital_Branding',
        
        # Circulaire Digital
        'CIRCULAIRE DIGITAL': 'Circulaire_Digital',
        'CIRCULAIRE DIGITALE': 'Circulaire_Digital',
        
        # Social Media
        'FACEBOOK + INSTAGRAM (PROMO)': 'Social_Media',
        'FACEBOOK + INSTAGRAM (PRODUIT)': 'Social_Media',
        'PINTEREST': 'Social_Media',
        'TIKTOK': 'Social_Media',
    }
    
    # Channels to exclude completely
    EXCLUDE_PATTERNS = [
        'PROGRAMMATIQUE',  # Includes GÉOCIBLAGE
        'AUDIO ET PODCAST',
        'ENVOIS POSTAUX',
        'COMMANDITES',
    ]
    
    # Skip patterns - headers, totals, subtotals
    SKIP_EXACT = ['FR', 'EN', 'TRADITIONNEL', 'NUMÉRIQUE', 'AUTRES']
    SKIP_CONTAINS = [
        'TOTAL', 'DIFFÉRENCE', '% VS', 
        'SEMAINE', 'CAMPAGNE', 'MEDIA', 'COOP', 'PRODUCTION', 'RÉSERVE',
        'CONTINGENCE', 'CIRCULAIRE PAPIER', 'RECHERCHE DE MOTS',
        'PREROLL - YOUTUBE', 'BANNIÈRES WEB',  # Subcomponent
    ]
    
    # Month mapping - fiscal year starts in November
    months_info = [
        ('NOVEMBRE', 11), ('DECEMBRE', 12), ('JANVIER', 1), ('FEVRIER', 2),
        ('MARS', 3), ('AVRIL', 4), ('MAI', 5), ('JUIN', 6),
        ('JUILLET', 7), ('AOUT', 8), ('SEPTEMBRE', 9), ('OCTOBRE', 10)
    ]
    
    # Find column indices for each month
    row6 = df_raw.iloc[6, :].tolist()
    month_cols = {}
    for i, val in enumerate(row6[:70]):
        if pd.notna(val):
            val_upper = str(val).upper().strip()
            for month_name, month_num in months_info:
                if val_upper == month_name and month_name not in month_cols:
                    month_cols[month_name] = (i, month_num)
                    break
    
    print(f"  Found {len(month_cols)} months for {year}")
    
    # Data rows to process
    data_rows = list(range(10, 37))
    
    media_data = []
    
    for row_idx in data_rows:
        media_name = df_raw.iloc[row_idx, 3]  # Column D
        
        if pd.isna(media_name) or not str(media_name).strip():
            continue
        
        media_name_str = str(media_name).strip()
        media_name_upper = media_name_str.upper()
        
        # Skip exact matches
        if media_name_upper in SKIP_EXACT:
            continue
        
        # Skip if contains certain patterns
        if any(skip in media_name_upper for skip in SKIP_CONTAINS):
            continue
        
        # Skip excluded channels
        if any(excl.upper() in media_name_upper for excl in EXCLUDE_PATTERNS):
            continue
        
        # For 2024, skip specific subcomponent
        if year == 2024 and media_name_upper == 'BANNIÈRES WEB':
            continue
        
        # Determine channel group
        channel_group = None
        for pattern, group in CHANNEL_GROUPS.items():
            if pattern in media_name_upper or media_name_upper == pattern:
                channel_group = group
                break
        
        # Skip if no group mapping found
        if channel_group is None:
            continue
        
        # Extract spend for each month
        for month_name, (col_idx, month_num) in month_cols.items():
            spend = df_raw.iloc[row_idx, col_idx]
            spend_value = pd.to_numeric(spend, errors='coerce')
            
            if pd.notna(spend_value) and spend_value != 0:
                media_data.append({
                    'year': year,
                    'month': month_name,
                    'month_num': month_num,
                    'channel_group': channel_group,
                    'spend': spend_value
                })
    
    df_clean = pd.DataFrame(media_data)
    
    if df_clean.empty:
        return df_clean
    
    # Aggregate by year, month, and channel group
    df_agg = df_clean.groupby(['year', 'month', 'month_num', 'channel_group'], 
                               as_index=False)['spend'].sum()
    
    return df_agg

# Process both budget files
print("Processing Budget 2024...")
budget_2024 = clean_budget_grouped(raw_path / 'Budget 2024 - REEL au 5 novembre.xlsx', 2024)

print("Processing Budget 2025...")
budget_2025 = clean_budget_grouped(raw_path / 'Budget 2025 - 21 août.xlsx', 2025)

# Combine both years
budget_combined = pd.concat([budget_2024, budget_2025], ignore_index=True)

# Create pivot for display
month_order = ['NOVEMBRE', 'DECEMBRE', 'JANVIER', 'FEVRIER', 'MARS', 'AVRIL', 
               'MAI', 'JUIN', 'JUILLET', 'AOUT', 'SEPTEMBRE', 'OCTOBRE']

print("\n" + "=" * 70)
print("BUDGET - 7 CHANNEL GROUPS")
print("=" * 70)

for year in [2024, 2025]:
    print(f"\n--- FISCAL YEAR {year} ---")
    year_data = budget_combined[budget_combined['year'] == year]
    if not year_data.empty:
        pivot = year_data.pivot_table(index='channel_group', columns='month', 
                                       values='spend', aggfunc='sum', fill_value=0)
        pivot = pivot[[m for m in month_order if m in pivot.columns]]
        print(pivot.round(0).to_string())
        print(f"\nTotal spend {year}: ${year_data['spend'].sum():,.0f}")
        
# Show channel totals across both years
print("\n" + "=" * 70)
print("TOTAL SPEND BY CHANNEL (2024-2025 Combined)")
print("=" * 70)
channel_totals = budget_combined.groupby('channel_group')['spend'].sum().sort_values(ascending=False)
for ch, total in channel_totals.items():
    pct = total / channel_totals.sum() * 100
    print(f"  {ch}: ${total:,.0f} ({pct:.1f}%)")
print(f"\n  TOTAL: ${channel_totals.sum():,.0f}")

In [ ]:
# Cell 3b: Create wide-format budget for merging with quotes

# Pivot to wide format: one row per year-month, columns for each channel
budget_wide = budget_combined.pivot_table(
    index=['year', 'month', 'month_num'],
    columns='channel_group',
    values='spend',
    aggfunc='sum',
    fill_value=0
).reset_index()

# Flatten column names
budget_wide.columns = ['year', 'month', 'month_num'] + \
    [f'spend_{col.lower().replace(" ", "_")}' for col in budget_wide.columns[3:]]

# Add total spend column
spend_cols = [c for c in budget_wide.columns if c.startswith('spend_')]
budget_wide['spend_total'] = budget_wide[spend_cols].sum(axis=1)

print("Wide-format budget created:")
print(f"  Shape: {budget_wide.shape}")
print(f"  Columns: {budget_wide.columns.tolist()}")

---
## 3. Tableau Medias 2025 - Campaign Performance

Extracts campaign-level metrics from the MASTER-TOTAL sheet.

**Columns extracted:**
- B (1): Date début - Campaign start date
- C (2): Date fin - Campaign end date  
- D (3): Média - Media type
- F (5): Station / Support - Media partner/platform
- L (11): Coût total ($ NET) - Total cost
- T (19): Nb occasions (RÉEL) - Actual number of spots
- U (20): Impressions totales (RÉEL) - Actual impressions
- V (21): PEB (RÉEL) - Actual GRPs
- AB (27): Vues complétées - Completed views
- AC (28): Taux de vues - View rate
- AD (29): Clics (RÉEL) - Actual clicks
- AE (30): Taux de clics - Click rate

**Note:** We also map to 6-channel groups for consistency

In [ ]:
# Cell 4: Clean Tableau Medias 2025

def clean_tableau_medias(file_path):
    """
    Clean the Tableau Medias file and map to 7 channel groups.
    """
    df_raw = pd.read_excel(file_path, sheet_name='MASTER-TOTAL', header=0)
    
    # Extract columns by index
    cols_to_extract = {
        1: 'date_debut',
        2: 'date_fin', 
        3: 'media_type',
        5: 'support',
        11: 'cost_net',
        19: 'occasions_reel',
        20: 'impressions_reel',
        21: 'peb_reel',
        27: 'vues_completees',
        28: 'taux_vues',
        29: 'clics_reel',
        30: 'taux_clics'
    }
    
    df_clean = df_raw.iloc[:, list(cols_to_extract.keys())].copy()
    df_clean.columns = list(cols_to_extract.values())
    
    # Remove empty rows
    df_clean = df_clean.dropna(how='all')
    df_clean = df_clean.dropna(subset=['date_debut', 'date_fin'], how='all')
    
    # Convert date columns
    df_clean['date_debut'] = pd.to_datetime(df_clean['date_debut'], errors='coerce')
    df_clean['date_fin'] = pd.to_datetime(df_clean['date_fin'], errors='coerce')
    
    # Convert numeric columns
    numeric_cols = ['cost_net', 'occasions_reel', 'impressions_reel', 'peb_reel',
                    'vues_completees', 'taux_vues', 'clics_reel', 'taux_clics']
    for col in numeric_cols:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
    
    # Map to 7 channel groups based on media_type and support
    def map_to_channel_group(row):
        media_type = str(row['media_type']).upper() if pd.notna(row['media_type']) else ''
        support = str(row['support']).upper() if pd.notna(row['support']) else ''
        
        if media_type == 'TÉLÉVISION':
            return 'Television'
        elif media_type == 'RADIO':
            return 'Audio'
        elif media_type == 'AFFICHAGE':
            return 'Affichage'
        elif media_type == 'NUMÉRIQUE':
            # Digital channels - use support to differentiate
            if 'GOOGLE' in support:
                return 'Search'
            elif 'CIRCULAIRE' in support or 'FLIPP' in support:
                return 'Circulaire_Digital'
            elif any(x in support for x in ['FACEBOOK', 'INSTAGRAM', 'PINTEREST', 'TIKTOK']):
                return 'Social_Media'
            else:
                # La Presse, Québecor, Bell preroll, RON = Digital Branding
                return 'Digital_Branding'
        else:
            return 'Other'
    
    df_clean['channel_group'] = df_clean.apply(map_to_channel_group, axis=1)
    
    # Add derived columns
    df_clean['year'] = df_clean['date_debut'].dt.year
    df_clean['month'] = df_clean['date_debut'].dt.month
    
    # Calculate CPM
    mask = (df_clean['impressions_reel'] > 0) & df_clean['cost_net'].notna()
    df_clean.loc[mask, 'cpm_calculated'] = (
        df_clean.loc[mask, 'cost_net'] / df_clean.loc[mask, 'impressions_reel']
    ) * 1000
    
    df_clean = df_clean.reset_index(drop=True)
    
    return df_clean

# Process the file
tableau_medias = clean_tableau_medias(raw_path / 'Recap_Tableau_Medias_2025.xlsx')

print("=" * 60)
print("TABLEAU MEDIAS 2025 - Campaign Performance")
print("=" * 60)
print(f"\nShape: {tableau_medias.shape}")
print(f"Date range: {tableau_medias['date_debut'].min()} to {tableau_medias['date_fin'].max()}")

print("\n--- By 7 Channel Groups ---")
summary = tableau_medias.groupby('channel_group').agg({
    'cost_net': 'sum',
    'impressions_reel': 'sum',
    'clics_reel': 'sum'
}).round(0)
print(summary.to_string())

---
## 4. Calendrier Fiscal (Fiscal Calendar)

Reference table for fiscal calendar mapping. The client's fiscal year starts in November.

In [ ]:
# Cell 5: Clean Calendrier Fiscal

def clean_calendrier_fiscal(file_path):
    """
    Clean the Calendrier Fiscal file.
    """
    df_raw = pd.read_excel(file_path, sheet_name='CalendrierFiscal', header=0)
    
    columns_to_keep = [
        'Date', 'Année', 'Mois', 'Nom Mois', 'Jour de la semaine',
        'Année fiscale', 'Trimestre', 'Semaine fiscale', 'Formule',
        'Semaine débutant le', 'Ordre du mois fiscal', 'Ordre semaine',
        'MoisFiscal', 'AnnéeNUM', 'Date début semaine'
    ]
    
    existing_cols = [col for col in columns_to_keep if col in df_raw.columns]
    df_clean = df_raw[existing_cols].copy()
    
    # Convert Date column
    if 'Date' in df_clean.columns:
        df_clean['Date'] = pd.to_datetime(df_clean['Date'], errors='coerce')
    
    # Remove rows where Date is NaN
    df_clean = df_clean.dropna(subset=['Date'])
    
    return df_clean

# Process the file
calendrier_fiscal = clean_calendrier_fiscal(raw_path / 'CalendrierFiscal.xlsx')

print("=" * 60)
print("CALENDRIER FISCAL")
print("=" * 60)
print(f"\nShape: {calendrier_fiscal.shape}")
print(f"Date range: {calendrier_fiscal['Date'].min()} to {calendrier_fiscal['Date'].max()}")
print(f"Fiscal years: {sorted(calendrier_fiscal['Année fiscale'].dropna().unique())}")

---
## 5. Save All Cleaned Datasets

In [ ]:
# Cell 6: Save all cleaned dataframes

# Save quotes
soumissions_combined.to_csv(processed_path / 'soumissions_quotes.csv', index=False)
soumissions_combined.to_pickle(processed_path / 'soumissions_quotes.pkl')

# Save budget - long format (for flexibility)
budget_combined.to_csv(processed_path / 'budget_media_spend.csv', index=False)
budget_combined.to_pickle(processed_path / 'budget_media_spend.pkl')

# Save budget - wide format (for merging)
budget_wide.to_csv(processed_path / 'budget_media_spend_wide.csv', index=False)
budget_wide.to_pickle(processed_path / 'budget_media_spend_wide.pkl')

# Save tableau medias
tableau_medias.to_csv(processed_path / 'tableau_medias_performance.csv', index=False)
tableau_medias.to_pickle(processed_path / 'tableau_medias_performance.pkl')

# Save calendrier fiscal
calendrier_fiscal.to_csv(processed_path / 'calendrier_fiscal.csv', index=False)
calendrier_fiscal.to_pickle(processed_path / 'calendrier_fiscal.pkl')

print("=" * 60)
print("FILES SAVED")
print("=" * 60)
print(f"\nSaved to: {processed_path}")
print("\nFiles created:")
for f in sorted(processed_path.glob('*')):
    size_kb = f.stat().st_size / 1024
    print(f"  - {f.name} ({size_kb:.1f} KB)")

---
## 6. Data Quality Summary

In [ ]:
# Cell 7: Data Quality Summary

print("=" * 70)
print("DATA QUALITY SUMMARY")
print("=" * 70)

datasets = {
    'Soumissions (Quotes)': soumissions_combined,
    'Budget (Media Spend - Long)': budget_combined,
    'Budget (Media Spend - Wide)': budget_wide,
    'Tableau Medias': tableau_medias,
    'Calendrier Fiscal': calendrier_fiscal
}

for name, df in datasets.items():
    print(f"\n{'-' * 40}")
    print(f"{name}")
    print(f"{'-' * 40}")
    print(f"  Rows: {len(df):,}")
    print(f"  Columns: {len(df.columns)}")
    
    total_cells = df.size
    missing_cells = df.isna().sum().sum()
    missing_pct = (missing_cells / total_cells) * 100
    print(f"  Missing values: {missing_cells:,} ({missing_pct:.1f}%)")

print("\n" + "=" * 70)
print("7 CHANNEL GROUPS FOR MMM")
print("=" * 70)
print("""
  1. Television        - TV (major brand investment)
  2. Audio             - Radio + Radio Numérique
  3. Affichage         - Billboards + Digital OOH
  4. Search            - Google Ads + Shopping
  5. Digital_Branding  - Premium display, La Presse, Preroll, Content
  6. Circulaire_Digital - Digital flyers
  7. Social_Media      - Facebook + Instagram + Pinterest + TikTok
  
  EXCLUDED: Programmatic, Audio/Podcast, Postal
""")

print("=" * 70)
print("NOTE: Missing values (NaN) preserved as per client instructions.")
print("=" * 70)